# ClinicalBERT Medication NER

Model: `pabRomero/BioClinicalBERT-full-finetuned-ner-pablo`

This notebook uses a BioClinicalBERT model fine-tuned for medication-related named entity recognition. It is useful for detecting medication mentions and medication context, not general clinical PHI entities like doctor, patient, hospital, or date.

## Supported Entities

| Entity | Meaning |
| --- | --- |
| `Drug` | Medication or drug name |
| `Reason` | Condition, symptom, or clinical reason for the medication |
| `Route` | Administration route, such as oral, IV, topical |
| `Strength` | Medication strength, such as 500 mg |
| `Form` | Medication form, such as tablet, capsule, solution |
| `Dosage` | Dose amount or dosing instruction |
| `Frequency` | How often the medication is taken |
| `Duration` | Treatment duration |
| `ADE` | Adverse drug event |

Internally the model uses BIO labels such as `B-Drug`, `I-Drug`, and `O`, but the notebook aggregates them into readable spans.

In [1]:
from __future__ import annotations

import pandas as pd
import torch
from transformers import AutoModelForTokenClassification, AutoTokenizer, pipeline

MODEL_NAME = "pabRomero/BioClinicalBERT-full-finetuned-ner-pablo"
device = 0 if torch.cuda.is_available() else -1

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME)

ner_pipeline = pipeline(
    task="token-classification",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple",
    device=device,
)

print(f"Loaded {MODEL_NAME} on {'cuda' if device == 0 else 'cpu'}.")
print("Raw BIO labels:")
print(model.config.id2label)

Device set to use cpu


Loaded pabRomero/BioClinicalBERT-full-finetuned-ner-pablo on cpu.
Raw BIO labels:
{0: 'O', 1: 'B-Drug', 2: 'I-Drug', 3: 'B-Reason', 4: 'I-Reason', 5: 'B-Route', 6: 'I-Route', 7: 'B-Strength', 8: 'I-Strength', 9: 'B-Form', 10: 'I-Form', 11: 'B-Dosage', 12: 'I-Dosage', 13: 'B-Frequency', 14: 'I-Frequency', 15: 'B-Duration', 16: 'I-Duration', 17: 'B-ADE', 18: 'I-ADE'}


Edit `input_text`, then run the detection cells.

In [5]:
input_text = "On 03/12/2019, Dr. Emily Smith at St. Mary's Hospital in Stuttgart prescribed ibuprofen 400 mg to 67-year-old Mr. David Jones, patient ID MRN-45892, for severe left knee pain caused by osteoarthritis after an X-ray showed joint-space narrowing."
input_text

"On 03/12/2019, Dr. Emily Smith at St. Mary's Hospital in Stuttgart prescribed ibuprofen 400 mg to 67-year-old Mr. David Jones, patient ID MRN-45892, for severe left knee pain caused by osteoarthritis after an X-ray showed joint-space narrowing."

In [6]:
def detect_medication_entities(text: str) -> pd.DataFrame:
    predictions = ner_pipeline(text)
    rows = []

    for prediction in predictions:
        rows.append(
            {
                "entity": prediction.get("entity_group", prediction.get("entity")),
                "text": prediction["word"],
                "start": prediction["start"],
                "end": prediction["end"],
                "score": round(float(prediction["score"]), 4),
            }
        )

    return pd.DataFrame(rows, columns=["entity", "text", "start", "end", "score"])


entities = detect_medication_entities(input_text)
entities

,entity,text,start,end,score
0,Drug,ibuprofen,78,87,0.9979
1,Strength,400 mg,88,94,0.9960
2,Strength,67,98,100,0.7042
3,Reason,severe,153,159,0.4966
4,Reason,knee pain,165,174,0.6732


In [7]:
if entities.empty:
    print("No medication-related entities detected.")
else:
    for row in entities.itertuples(index=False):
        print(f"{row.entity}: {row.text} ({row.score})")

Drug: ibuprofen (0.9979)
Strength: 400 mg (0.996)
Strength: 67 (0.7042)
Reason: severe (0.4966)
Reason: knee pain (0.6732)


Expected behavior on the default sentence: the model should detect `ibuprofen` as `Drug` and may detect `severe knee pain` as `Reason`. It should not be expected to detect `Dr. Smith`, `Mr. Jones`, `St. Mary's Hospital`, or `03/12/2019` because those labels are not part of this model's output space.